# Publication copy: portable paths and cleared outputs

Run the path configuration below before the original notebook cells. Change only PROJECT_ROOT. On Windows use a raw string such as r"D:\Phishing_Paper_Reproducibility". The data filenames and results/revision layout are preserved.

This copy has cleared cell outputs and execution counts. Keep the executed original and its saved results separately. This path revision is not a new scientific run or a full methodological audit.

**Historical extraction/development notebook:** includes repeated extraction cells, an exploratory MMD calculation and a Char-CNN experiment. Do not present every cell as the finalized paper pipeline. Extraction cells overwrite the named unified CSVs when explicitly run; use a disposable data copy for regeneration. Do not run this notebook merely to inspect saved analysis.


In [ ]:
# Publication path configuration: edit only PROJECT_ROOT.
from pathlib import Path, PurePosixPath
import sys

# Empty: infer the root when opened from the project root or notebooks/.
# Otherwise enter your local absolute folder, or a /content/drive/... folder.
PROJECT_ROOT = ""
if not PROJECT_ROOT:
    candidates = [Path.cwd(), Path.cwd().parent]
    found = [p for p in candidates if (p / "data").is_dir() and (p / "notebooks").is_dir()]
    if len(found) != 1:
        raise RuntimeError("Set PROJECT_ROOT to your Phishing_Paper_Reproducibility folder.")
    PROJECT_ROOT = found[0]
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser()
if str(PROJECT_ROOT).startswith("/content/drive/"):
    if "google.colab" not in sys.modules:
        raise RuntimeError("A Drive root requires Colab. Use a local folder on your computer.")
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
PROJECT_ROOT = PROJECT_ROOT.resolve()
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project root missing: {PROJECT_ROOT}")

DATA_FILENAMES = {
    "phiusiil.csv", "phiusiil_unified.csv", "URL-Phish.csv",
    "urlphish_unified.csv", "LegitPhish Dataset.csv", "legitphish_unified.csv",
}
def relocated_dataset_path(recorded_path):
    # Immutable historical records are kept byte-for-byte. Map only the
    # approved data members to this root; existing old Drive files are ignored.
    old = PurePosixPath(str(recorded_path).replace("\\", "/"))
    if old.parent.name != "data" or old.name not in DATA_FILENAMES:
        raise ValueError(f"Unrecognized recorded dataset: {recorded_path}")
    return PROJECT_ROOT / "data" / old.name

print("Project root:", PROJECT_ROOT)


In [ ]:
# Intention: Install robust parsing dependencies
!pip install tldextract tqdm

In [ ]:
import pandas as pd
import tldextract
import re
from tqdm.auto import tqdm
from urllib.parse import urlparse

class UniversalURLFeatureExtractor:
    def __init__(self):
        # Initialize tldextract to silently fetch TLD lists including private domains
        self.extract = tldextract.TLDExtract(include_psl_private_domains=True)

    def extract_features(self, url):
        """Extracts exact numeric and categorical features from a single URL string."""
        url = str(url).strip()
        url_len = len(url)

        # Fallback for empty rows
        if url_len == 0:
            return self._empty_feature_dict()

        # Parse with urllib and tldextract
        # Force a scheme if missing so netloc parses correctly
        parsed = urlparse(url if "//" in url else f"http://{url}")
        ext = self.extract(url)

        # Base properties
        tld = ext.suffix
        subdomain = ext.subdomain

        # Character distributions
        num_letters = sum(c.isalpha() for c in url)
        num_digits = sum(c.isdigit() for c in url)
        num_equals = url.count('=')
        num_qmark = url.count('?')
        num_ampersand = url.count('&')

        # 'Other Special Chars' is the total non-alphanumeric minus specific trackers
        total_special = sum(not c.isalnum() for c in url)
        num_other_special = total_special - (num_equals + num_qmark + num_ampersand)

        # Construct dictionary mirroring exact column order from the source
        features = {
            'URLLength': url_len,
            'DomainLength': len(parsed.netloc),
            'IsDomainIP': 1 if self._is_ip(parsed.hostname or "") else 0,
            'TLD': tld,
            'TLDLength': len(tld),
            'NoOfSubDomain': len(subdomain.split('.')) if subdomain else 0,
            'NoOfLettersInURL': num_letters,
            'LetterRatioInURL': num_letters / url_len,
            'NoOfDegitsInURL': num_digits,
            'DegitRatioInURL': num_digits / url_len,
            'NoOfEqualsInURL': num_equals,
            'NoOfQMarkInURL': num_qmark,
            'NoOfAmpersandInURL': num_ampersand,
            'NoOfOtherSpecialCharsInURL': num_other_special,
            'IsHTTPS': 1 if parsed.scheme.lower() == 'https' else 0
        }

        return features

    def _is_ip(self, hostname):
        """Checks if the parsed hostname is an IPv4 address."""
        ip_pattern = re.compile(r"^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$")
        return bool(ip_pattern.match(hostname))

    def _empty_feature_dict(self):
        """Standardized fallback to prevent pipeline crashes."""
        return {
            'URLLength': 0, 'DomainLength': 0, 'IsDomainIP': 0, 'TLD': '',
            'TLDLength': 0, 'NoOfSubDomain': 0, 'NoOfLettersInURL': 0,
            'LetterRatioInURL': 0.0, 'NoOfDegitsInURL': 0, 'DegitRatioInURL': 0.0,
            'NoOfEqualsInURL': 0, 'NoOfQMarkInURL': 0, 'NoOfAmpersandInURL': 0,
            'NoOfOtherSpecialCharsInURL': 0, 'IsHTTPS': 0
        }

    def transform_dataset(self, df, url_column, label_column):
        """Iterates through the dataset, extracts features, and binds the target label."""
        extracted_rows = []

        # Wraps execution in tqdm for the progress bar rendering
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extracting features from {len(df)} URLs"):
            feature_dict = self.extract_features(row[url_column])

            # Dynamically attach the harmonization label from the source dataframe
            feature_dict[label_column] = row[label_column]
            extracted_rows.append(feature_dict)

        return pd.DataFrame(extracted_rows)

In [ ]:
# Intention: Robustly load datasets and dynamically harmonize target labels
import pandas as pd
import os

# 1. Mount Google Drive securely

# 2. Define the exact Q1 project paths
BASE_DATA_DIR = str(PROJECT_ROOT / 'data')

phiusiil_path = os.path.join(BASE_DATA_DIR, 'phiusiil.csv')
urlphish_path = os.path.join(BASE_DATA_DIR, 'URL-Phish.csv')
legitphish_path = os.path.join(BASE_DATA_DIR, 'LegitPhish Dataset.csv')

if not os.path.exists(urlphish_path):
    urlphish_path = os.path.join(BASE_DATA_DIR, 'URL-Phish dataset.csv')

print("Loading raw datasets from Drive...")
df_phi_raw = pd.read_csv(phiusiil_path)
df_urlphish_raw = pd.read_csv(urlphish_path)
df_legitphish_raw = pd.read_csv(legitphish_path)

# --- Diagnostic Step ---
print("\n[Diagnostic] URL-Phish Columns:", df_urlphish_raw.columns.tolist())

# 3. Defensive Target Identification
def find_target_column(df, dataset_name):
    """Dynamically identifies the target label column based on common naming conventions."""
    candidates = ['label', 'Label', 'class', 'Class', 'ClassLabel', 'target', 'Target', 'result', 'Result']
    for col in candidates:
        if col in df.columns:
            print(f"[{dataset_name}] Target column identified as: '{col}'")
            return col
    raise KeyError(f"Could not identify target column in {dataset_name}. Available columns: {df.columns.tolist()}")

# Initialize the Q1-standard universal extractor
extractor = UniversalURLFeatureExtractor()

print("\n--- Processing Source Benchmark: PhiUSIIL ---")
phi_target = find_target_column(df_phi_raw, "PhiUSIIL")
df_phi_raw['harmonized_label'] = df_phi_raw[phi_target]
df_phi_unified = extractor.transform_dataset(df_phi_raw, url_column='URL', label_column='harmonized_label')

print("\n--- Processing Target Benchmark 1: URL-Phish ---")
# Dynamically locate the target column (which caused the previous KeyError)
urlphish_target = find_target_column(df_urlphish_raw, "URL-Phish")
df_urlphish_raw['harmonized_label'] = df_urlphish_raw[urlphish_target]
# Ensure we map the correct URL column name ('url' vs 'URL' vs 'Url')
urlphish_url_col = 'url' if 'url' in df_urlphish_raw.columns else 'URL'
df_urlphish_unified = extractor.transform_dataset(df_urlphish_raw, url_column=urlphish_url_col, label_column='harmonized_label')

print("\n--- Processing Target Benchmark 2: LegitPhish ---")
legitphish_target = find_target_column(df_legitphish_raw, "LegitPhish")
# LegitPhish natively utilizes 0 = phishing, 1 = legitimate; requires inversion
df_legitphish_raw['harmonized_label'] = df_legitphish_raw[legitphish_target].apply(lambda x: 1 if x == 0 else 0)
df_legitphish_unified = extractor.transform_dataset(df_legitphish_raw, url_column='URL', label_column='harmonized_label')

# 4. Save the strictly aligned datasets
out_phi = os.path.join(BASE_DATA_DIR, 'phiusiil_unified.csv')
out_urlphish = os.path.join(BASE_DATA_DIR, 'urlphish_unified.csv')
out_legitphish = os.path.join(BASE_DATA_DIR, 'legitphish_unified.csv')

print("\nSaving universally extracted datasets to Drive...")
df_phi_unified.to_csv(out_phi, index=False)
df_urlphish_unified.to_csv(out_urlphish, index=False)
df_legitphish_unified.to_csv(out_legitphish, index=False)

print("\nExtraction complete. Methodological parity achieved.")

In [ ]:
import os
for root, dirs, files in os.walk(str(PROJECT_ROOT)):
    for f in files:
        if f.endswith(('.py', '.ipynb')):
            print(os.path.join(root, f))

In [ ]:
import pandas as pd
import tldextract
import re
from tqdm.auto import tqdm
from urllib.parse import urlparse

class UniversalURLFeatureExtractor:
    def __init__(self):
        # Initialize tldextract to silently fetch TLD lists including private domains
        self.extract = tldextract.TLDExtract(include_psl_private_domains=True)

    def extract_features(self, url):
        """Extracts exact numeric and categorical features from a single URL string."""
        url = str(url).strip()
        url_len = len(url)

        # Fallback for empty rows
        if url_len == 0:
            return self._empty_feature_dict()

        # Parse with urllib and tldextract
        # Force a scheme if missing so netloc parses correctly
        parsed = urlparse(url if "//" in url else f"http://{url}")
        ext = self.extract(url)

        # Base properties
        tld = ext.suffix
        subdomain = ext.subdomain

        # Character distributions
        num_letters = sum(c.isalpha() for c in url)
        num_digits = sum(c.isdigit() for c in url)
        num_equals = url.count('=')
        num_qmark = url.count('?')
        num_ampersand = url.count('&')

        # 'Other Special Chars' is the total non-alphanumeric minus specific trackers
        total_special = sum(not c.isalnum() for c in url)
        num_other_special = total_special - (num_equals + num_qmark + num_ampersand)

        # Construct dictionary mirroring exact column order from the source
        features = {
            'URLLength': url_len,
            'DomainLength': len(parsed.netloc),
            'IsDomainIP': 1 if self._is_ip(parsed.hostname or "") else 0,
            'TLD': tld,
            'TLDLength': len(tld),
            'NoOfSubDomain': len(subdomain.split('.')) if subdomain else 0,
            'NoOfLettersInURL': num_letters,
            'LetterRatioInURL': num_letters / url_len,
            'NoOfDegitsInURL': num_digits,
            'DegitRatioInURL': num_digits / url_len,
            'NoOfEqualsInURL': num_equals,
            'NoOfQMarkInURL': num_qmark,
            'NoOfAmpersandInURL': num_ampersand,
            'NoOfOtherSpecialCharsInURL': num_other_special,
            'IsHTTPS': 1 if parsed.scheme.lower() == 'https' else 0
        }

        return features

    def _is_ip(self, hostname):
        """Checks if the parsed hostname is an IPv4 address."""
        ip_pattern = re.compile(r"^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$")
        return bool(ip_pattern.match(hostname))

    def _empty_feature_dict(self):
        """Standardized fallback to prevent pipeline crashes."""
        return {
            'URLLength': 0, 'DomainLength': 0, 'IsDomainIP': 0, 'TLD': '',
            'TLDLength': 0, 'NoOfSubDomain': 0, 'NoOfLettersInURL': 0,
            'LetterRatioInURL': 0.0, 'NoOfDegitsInURL': 0, 'DegitRatioInURL': 0.0,
            'NoOfEqualsInURL': 0, 'NoOfQMarkInURL': 0, 'NoOfAmpersandInURL': 0,
            'NoOfOtherSpecialCharsInURL': 0, 'IsHTTPS': 0
        }

    def transform_dataset(self, df, url_column, label_column):
        """Iterates through the dataset, extracts features, and binds the target label."""
        extracted_rows = []

        # Wraps execution in tqdm for the progress bar rendering
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extracting features from {len(df)} URLs"):
            feature_dict = self.extract_features(row[url_column])

            # Dynamically attach the harmonization label from the source dataframe
            feature_dict[label_column] = row[label_column]
            extracted_rows.append(feature_dict)

        return pd.DataFrame(extracted_rows)

In [ ]:
# Intention: Mount Google Drive, load raw datasets, and execute uniform extraction
import pandas as pd
import os

# 1. Mount Google Drive securely

# 2. Define the exact project paths
BASE_DATA_DIR = str(PROJECT_ROOT / 'data')

phiusiil_path = os.path.join(BASE_DATA_DIR, 'phiusiil.csv')
urlphish_path = os.path.join(BASE_DATA_DIR, 'URL-Phish.csv')
legitphish_path = os.path.join(BASE_DATA_DIR, 'LegitPhish Dataset.csv')

# Fallback for URL-Phish naming variance noted in your methodology
if not os.path.exists(urlphish_path):
    urlphish_path = os.path.join(BASE_DATA_DIR, 'URL-Phish dataset.csv')

# 3. Load datasets into memory
print("Loading raw datasets from Drive...")
df_phi_raw = pd.read_csv(phiusiil_path)
df_urlphish_raw = pd.read_csv(urlphish_path)
df_legitphish_raw = pd.read_csv(legitphish_path)

# Initialize the Q1-standard universal extractor (from the previous cell)
extractor = UniversalURLFeatureExtractor()

print("\n--- Processing Source Benchmark: PhiUSIIL ---")
# The project methodology enforces 1 = phishing, 0 = legitimate
df_phi_raw['harmonized_label'] = df_phi_raw['label']
df_phi_unified = extractor.transform_dataset(df_phi_raw, url_column='URL', label_column='harmonized_label')

print("\n--- Processing Target Benchmark 1: URL-Phish ---")
# URL-Phish natively conforms to 1 = phishing, 0 = legitimate
df_urlphish_raw['harmonized_label'] = df_urlphish_raw['label']
df_urlphish_unified = extractor.transform_dataset(df_urlphish_raw, url_column='url', label_column='harmonized_label')

print("\n--- Processing Target Benchmark 2: LegitPhish ---")
# LegitPhish natively utilizes 0 = phishing, 1 = legitimate; requires inversion
df_legitphish_raw['harmonized_label'] = df_legitphish_raw['ClassLabel'].apply(lambda x: 1 if x == 0 else 0)
df_legitphish_unified = extractor.transform_dataset(df_legitphish_raw, url_column='URL', label_column='harmonized_label')

# 4. Save the strictly aligned, methodologically defensible datasets
out_phi = os.path.join(BASE_DATA_DIR, 'phiusiil_unified.csv')
out_urlphish = os.path.join(BASE_DATA_DIR, 'urlphish_unified.csv')
out_legitphish = os.path.join(BASE_DATA_DIR, 'legitphish_unified.csv')

print("\nSaving universally extracted datasets to Drive...")
df_phi_unified.to_csv(out_phi, index=False)
df_urlphish_unified.to_csv(out_urlphish, index=False)
df_legitphish_unified.to_csv(out_legitphish, index=False)

print("\nExtraction complete. Methodological parity achieved.")

In [ ]:
# Intention: Quantify zero-inflation (sparsity) across datasets for Q1 methodology reporting
import pandas as pd
import numpy as np

# List of the sparse features in question
sparse_features = [
    'NoOfDegitsInURL', 'DegitRatioInURL',
    'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL'
]

def calculate_sparsity(df, features, dataset_name):
    """Calculates the percentage of exactly 0.0 values for given features."""
    sparsity_data = []
    for col in features:
        zero_count = (df[col] == 0).sum()
        total_count = len(df)
        zero_pct = (zero_count / total_count) * 100
        sparsity_data.append({'Feature': col, f'{dataset_name} % Zeros': np.round(zero_pct, 2)})
    return pd.DataFrame(sparsity_data).set_index('Feature')

# Assuming df_phi_unified, df_urlphish_unified, df_legitphish_unified are still in memory
sparsity_phi = calculate_sparsity(df_phi_unified, sparse_features, "PhiUSIIL")
sparsity_url = calculate_sparsity(df_urlphish_unified, sparse_features, "URL-Phish")
sparsity_legit = calculate_sparsity(df_legitphish_unified, sparse_features, "LegitPhish")

# Concatenate for a publication-ready table
df_sparsity_table = pd.concat([sparsity_phi, sparsity_url, sparsity_legit], axis=1)
print("\n--- Feature Sparsity (Zero-Inflation) Table ---")
display(df_sparsity_table)

In [ ]:
# Intention: Formalize Covariate Shift via MMD for Q1 Methodology Reporting
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel
import os

# 1. Configuration
BASE_DATA_DIR = str(PROJECT_ROOT / 'data')
SAMPLE_SIZE = 5000
RANDOM_SEED = 42

print("Loading universally extracted datasets...")
df_phi = pd.read_csv(os.path.join(BASE_DATA_DIR, 'phiusiil_unified.csv'))
df_urlphish = pd.read_csv(os.path.join(BASE_DATA_DIR, 'urlphish_unified.csv'))
df_legitphish = pd.read_csv(os.path.join(BASE_DATA_DIR, 'legitphish_unified.csv'))

# 2. Isolate the 14 Continuous & Binary Features (excluding categorical 'TLD')
numeric_features = [
    'URLLength', 'DomainLength', 'IsDomainIP', 'TLDLength', 'NoOfSubDomain',
    'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL',
    'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL', 'IsHTTPS'
]

def extract_and_scale(df, features, sample_size, seed):
    """Subsamples and standardizes features for RKHS mapping."""
    df_sample = df.sample(n=min(len(df), sample_size), random_state=seed)
    X = df_sample[features].values
    scaler = StandardScaler()
    return scaler.fit_transform(X)

print(f"Stratifying and scaling {SAMPLE_SIZE} instances per dataset...")
X_source = extract_and_scale(df_phi, numeric_features, SAMPLE_SIZE, RANDOM_SEED)
X_target1 = extract_and_scale(df_urlphish, numeric_features, SAMPLE_SIZE, RANDOM_SEED)
X_target2 = extract_and_scale(df_legitphish, numeric_features, SAMPLE_SIZE, RANDOM_SEED)

# 3. Vectorized MMD Computation
def compute_mmd(X, Y, gamma=None):
    """Computes the unbiased empirical estimate of MMD^2 using an RBF kernel."""
    if gamma is None:
        gamma = 1.0 / X.shape[1] # Median heuristic analog

    K_XX = rbf_kernel(X, X, gamma=gamma)
    K_YY = rbf_kernel(Y, Y, gamma=gamma)
    K_XY = rbf_kernel(X, Y, gamma=gamma)

    n, m = K_XX.shape[0], K_YY.shape[0]

    # Remove diagonal terms for the unbiased estimate
    np.fill_diagonal(K_XX, 0)
    np.fill_diagonal(K_YY, 0)

    term_XX = np.sum(K_XX) / (n * (n - 1))
    term_YY = np.sum(K_YY) / (m * (m - 1))
    term_XY = np.sum(K_XY) / (n * m)

    return term_XX + term_YY - 2 * term_XY

# 4. Execute Quantifications
print("\n--- Computing Covariate Shift (MMD^2) ---")
mmd_phi_vs_urlphish = compute_mmd(X_source, X_target1)
mmd_phi_vs_legitphish = compute_mmd(X_source, X_target2)

print(f"MMD^2 (PhiUSIIL -> URL-Phish):  {mmd_phi_vs_urlphish:.6f}")
print(f"MMD^2 (PhiUSIIL -> LegitPhish): {mmd_phi_vs_legitphish:.6f}")

In [ ]:
# Intention: Train a Char-CNN Baseline with Strict RAM Management (OOM-Safe)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
import gc
from tqdm.auto import tqdm

# 1. Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

# 2. Hyperparameters & Paths
BASE_DATA_DIR = str(PROJECT_ROOT / 'data')
MAX_LEN = 200
BATCH_SIZE = 512
EPOCHS = 3
LEARNING_RATE = 1e-3

phiusiil_path = os.path.join(BASE_DATA_DIR, 'phiusiil.csv')
urlphish_path = os.path.join(BASE_DATA_DIR, 'URL-Phish.csv')
if not os.path.exists(urlphish_path):
    urlphish_path = os.path.join(BASE_DATA_DIR, 'URL-Phish dataset.csv')
legitphish_path = os.path.join(BASE_DATA_DIR, 'LegitPhish Dataset.csv')

# 3. Memory-Safe Data Loader Helper
def load_minimal_dataset(path, dataset_name):
    """Loads strictly the URL and Target columns to prevent Colab OOM crashes."""
    # Read only the first row to inspect column names
    df_head = pd.read_csv(path, nrows=1)

    # Identify target column dynamically
    candidates = ['label', 'Label', 'class', 'Class', 'ClassLabel', 'target', 'Target', 'result', 'Result']
    target_col = next((col for col in candidates if col in df_head.columns), None)
    if not target_col:
        raise KeyError(f"Could not identify target column in {dataset_name}.")

    # Identify URL column dynamically
    url_col = 'url' if 'url' in df_head.columns else 'URL'

    print(f"[{dataset_name}] Loading explicitly: '{url_col}' and '{target_col}'...")
    # Load ONLY the required columns
    df_minimal = pd.read_csv(path, usecols=[url_col, target_col])
    return df_minimal, url_col, target_col

# 4. Neural Architecture & Dataset Definitions
chars = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 -,;.!?:'\"/\\|_@#$%^&*~`+-=<>()[]{}"
char2idx = {c: i + 1 for i, c in enumerate(chars)}
vocab_size = len(char2idx) + 1

class RawURLDataset(Dataset):
    def __init__(self, urls, labels, max_len):
        self.urls = urls.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.max_len = max_len

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        url = str(self.urls.iloc[idx])
        label = float(self.labels.iloc[idx])
        sequence = [char2idx.get(c, 0) for c in url[:self.max_len]]
        if len(sequence) < self.max_len:
            sequence += [0] * (self.max_len - len(sequence))
        return torch.tensor(sequence, dtype=torch.long), torch.tensor(label, dtype=torch.float)

class CharCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, num_filters=256, kernel_size=5):
        super(CharCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.conv1d = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=kernel_size)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters, 1)

    def forward(self, x):
        embedded = self.embedding(x).transpose(1, 2)
        conved = self.relu(self.conv1d(embedded))
        pooled = self.pool(conved).squeeze(2)
        dropped = self.dropout(pooled)
        return self.fc(dropped).squeeze(1)

# 5. Phase A: Train on Source Only
df_phi, phi_url_col, phi_tgt_col = load_minimal_dataset(phiusiil_path, "PhiUSIIL")
train_dataset = RawURLDataset(df_phi[phi_url_col], df_phi[phi_tgt_col], MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

model = CharCNN(vocab_size=vocab_size).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\n--- Training Char-CNN on PhiUSIIL ---")
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    correct = 0
    total = 0
    for seqs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        seqs, labels = seqs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(seqs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f} | Train Acc: {correct/total:.4f}")

# 6. Free up RAM before Inference
del df_phi
del train_loader
del train_dataset
gc.collect()

# 7. Phase B: Zero-Shot Inference
def evaluate_saturation(path, name, label_inversion=False):
    df_tgt, url_col, tgt_col = load_minimal_dataset(path, name)
    labels = df_tgt[tgt_col].apply(lambda x: 1 if x == 0 else 0) if label_inversion else df_tgt[tgt_col]

    dataset = RawURLDataset(df_tgt[url_col], labels, MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    model.eval()
    all_probs = []
    with torch.no_grad():
        for seqs, _ in loader:
            probs = torch.sigmoid(model(seqs.to(device))).cpu().numpy()
            all_probs.extend(probs)

    all_probs = np.array(all_probs)
    print(f"\n--- Char-CNN Zero-Shot Inference: {name} ---")
    print(f"Mean Proba:       {np.mean(all_probs):.6f}")
    print(f"Median Proba:     {np.median(all_probs):.6f}")
    print(f"Fraction >= 0.99: {np.mean(all_probs >= 0.99):.6f}")

    # Cleanup memory for next dataset
    del df_tgt
    del dataset
    del loader
    gc.collect()

print("\n--- Evaluating External Target Domains ---")
evaluate_saturation(urlphish_path, "URL-Phish", label_inversion=False)
evaluate_saturation(legitphish_path, "LegitPhish", label_inversion=True)

In [ ]:
# Intention: Hot-patch the tokenizer and resume zero-shot inference without retraining
import gc

# 1. Patch the out-of-bounds index
if '}' in char2idx and char2idx['}'] >= vocab_size:
    char2idx['}'] = 0  # Map to safe padding/unknown token

print("Tokenizer patched successfully. Resuming neural inference...")

# 2. Re-run Phase B Evaluations using the model already in memory
def evaluate_saturation(path, name, label_inversion=False):
    df_tgt, url_col, tgt_col = load_minimal_dataset(path, name)
    labels = df_tgt[tgt_col].apply(lambda x: 1 if x == 0 else 0) if label_inversion else df_tgt[tgt_col]

    dataset = RawURLDataset(df_tgt[url_col], labels, MAX_LEN)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    model.eval()
    all_probs = []
    with torch.no_grad():
        for seqs, _ in loader:
            probs = torch.sigmoid(model(seqs.to(device))).cpu().numpy()
            all_probs.extend(probs)

    all_probs = np.array(all_probs)
    print(f"\n--- Char-CNN Zero-Shot Inference: {name} ---")
    print(f"Mean Proba:       {np.mean(all_probs):.6f}")
    print(f"Median Proba:     {np.median(all_probs):.6f}")
    print(f"Fraction >= 0.99: {np.mean(all_probs >= 0.99):.6f}")

    # Cleanup
    del df_tgt, dataset, loader
    gc.collect()

print("\n--- Evaluating External Target Domains ---")
evaluate_saturation(urlphish_path, "URL-Phish", label_inversion=False)
evaluate_saturation(legitphish_path, "LegitPhish", label_inversion=True)